In [19]:
import pandas as pd
data1 = pd.read_csv("processed_data.csv")
data2 = pd.read_csv("newfile.csv")
print(data1.columns)
print(data2.columns)

Index(['articleID', 'domain', 'date', 'category', 'label', 'source', 'F-type',
       'text'],
      dtype='object')
Index(['label', 'text'], dtype='object')


In [3]:
print(data1['label'].value_counts())

label
1    7202
0    1299
Name: count, dtype: int64


In [4]:
print(data2['label'].value_counts())

label
0        5836
1        5836
label       1
Name: count, dtype: int64


In [5]:
from sklearn.utils import resample

data1_majority = data1[data1.label == 1]
data1_minority = data1[data1.label == 0]

data1_majority_down = resample(data1_majority,
                            replace=False,
                            n_samples=1299,
                            random_state=42)

data1_balanced = pd.concat([data1_majority_down, data1_minority])

print(data1_balanced['label'].value_counts())


label
1    1299
0    1299
Name: count, dtype: int64


In [6]:
merged_data = pd.concat([data1_balanced, data2], ignore_index=True)

In [7]:
print(merged_data['label'].value_counts())

label
1        5836
0        5836
1        1299
0        1299
label       1
Name: count, dtype: int64


In [ ]:
#fixing the extra "label" named row and converting strings 0 or 1 to int
merged_data['label'] = merged_data['label'].astype(str) 
merged_data = merged_data[merged_data['label'].isin(['0', '1'])] 
merged_data['label'] = merged_data['label'].astype(int)  



In [15]:

print(merged_data.shape)
print(merged_data['label'].value_counts())


(14270, 8)
label
1    7135
0    7135
Name: count, dtype: int64


In [16]:
merged_data.to_csv("New_dataset.csv", index=False)


In [18]:
new_data = pd.read_csv("New_dataset.csv")
print(new_data.columns)

Index(['articleID', 'domain', 'date', 'category', 'label', 'source', 'F-type',
       'text'],
      dtype='object')


In [ ]:
new_data = new_data[['text', 'label']]
print(new_data.shape)

(14270, 2)


In [25]:
new_data.head(5)

,text,label
0,ঝিনাইদহে দুই পক্ষের গোলাগুলিতে নিহত ১ ঝিনাইদহে...,1
1,ফরিদপুরে ‘অশ্লীল’ ভিডিওসহ আটক ৫ র‌্যাব ৮-এর ফর...,1
2,কারাগার থেকে মুক্ত হলেন নওয়াজ ও তার মেয়ে ১৯ সে...,1
3,রোহিঙ্গা নারীকে পাচারের অভিযোগে একজনকে কারাদণ্...,1
4,শার্শায় গুচ্ছ গ্রামের স্থান পরিদর্শনে উপসচিব অ...,1


In [49]:
new_data=pd.read_csv("New_dataset_preprocessed.csv")


print(new_data.isnull().sum())


text     0
label    0
dtype: int64


In [55]:
import pandas as pd
import pickle
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load new dataset
new_data = pd.read_csv("New_dataset.csv")
new_data = new_data[['text','label']].dropna()

# Load TF-IDF vectorizer
with open("tfidf_vectorizer.pkl","rb") as f:
    vectorizer = pickle.load(f)

X_new = vectorizer.transform(new_data['text'])
y_true = new_data['label'].values

# Load your trained XGBoost or RandomForest model
with open("xgb_model.pkl", "rb") as f:  # or rf_model.pkl
    model = pickle.load(f)

# Predict
y_pred = model.predict(X_new)

# Evaluate
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("Classification Report:\n", classification_report(y_true, y_pred))


Accuracy: 0.7505080944705306
Confusion Matrix:
 [[5751 1384]
 [2176 4958]]
Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.81      0.76      7135
           1       0.78      0.69      0.74      7134

    accuracy                           0.75     14269
   macro avg       0.75      0.75      0.75     14269
weighted avg       0.75      0.75      0.75     14269



In [ ]:
import pandas as pd
import re
import string


bangla_normalization_dict = {
    "মুরগি": "মুরগি",
    "মুরগী": "মুরগি",
    "মুরগীটা": "মুরগি",
    "মুরগিটা": "মুরগি",
    "মুরগিগুলো": "মুরগি",
    "শিয়াল": "শিয়াল",
    "শেয়াল": "শিয়াল",
    "শিয়াল": "শিয়াল",
    "মানুষ": "মানুষ",
    "মাঞ্জুষ": "মানুষ",
    "মানুষটা": "মানুষ",
    "ইন্টারভিউ": "ইন্টারভিউ",
    "ক্যামেরাম্যান": "ক্যামেরাম্যান",
    "মডেল": "মডেল",
    "টিম": "টিম",
    "বস": "বস",
    "ভিডি": "ভিডিও",
    "ফরান্স": "ফ্রান্স",
    "খাচ্ছে": "খাওয়া",
    "খাচ্ছেন": "খাওয়া",
    "খায়": "খাওয়া",
    "খেতে": "খাওয়া",
    "খেয়েছে": "খাওয়া",
    "দেয়": "দেওয়া",
    "দিচ্ছে": "দেওয়া",
    "দিয়েছে": "দেওয়া",
    "একটি": "এক",
    "এক": "এক",
    "দুইটি": "দুই",
    "দুই": "দুই",
}

# Removes the suffixes
def simple_bangla_stem(word):
    suffixes = ["গুলো", "দের", "মতো", "গুলোই", "টিতে", "টার", "রা", "টি", "টা", "তে", "র", "রা", "এর", "ই", "ও"]
    for suf in sorted(suffixes, key=len, reverse=True):
        if word.endswith(suf) and len(word) > len(suf) + 1:
            return word[:-len(suf)]
    return word

#  stopwords for HAN
han_stopwords = ['এবং', 'কিন্তু', 'তবে', 'ও', 'এর', 'এটি', 'এই']

# Splits whenever there is dari or a newline
def sentence_tokenize(text):
    sentences = [s.strip() for s in re.split(r'।|\n', text) if s.strip()]
    return sentences

#  cleaning
def tokenize_words(sentence):
    words = sentence.split()
    words = [bangla_normalization_dict.get(w, w) for w in words] # Returns the normalized form if it does exist in normalization dictionary

    words = [simple_bangla_stem(w) for w in words]
    # Remove  stopwords
    words = [w for w in words if w not in han_stopwords]
    # Remove quotes/punctuation
    words = [re.sub(r"[‘’“”…'\"`]", "", w) for w in words if re.sub(r"[‘’“”…'\"`]", "", w)]

    # If it finds any English word, convert to Lowercase
    words = [w.lower() if w.isalpha() and w.isascii() else w for w in words]
    return words

#  HAN preprocessing
def preprocess_for_HAN(text):
    text = str(text)
    # Remove HTML, URLs, emails, hashtags
    text = re.sub(r"<.*?>|http\S+|www\S+|\S*@\S*|\#\S*", " ", text)
    # Remove Latin letters, digits,  punctuation (except | )
    text = re.sub(r"[a-zA-Z0-9!@#$%^&*(),\"\'\-\‘\’\“\”]+", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Sentence tokenize
    sentences = sentence_tokenize(text)
    # Word tokenize + cleaning
    processed_sentences = [tokenize_words(sent) for sent in sentences]
    # Remove empty sentences
    processed_sentences = [s for s in processed_sentences if len(s) > 0]
    return processed_sentences

processed_data = pd.read_csv("newfile.csv")
processed_data['processed_text_han'] = processed_data['text'].apply(preprocess_for_HAN)

# Checking
for original, han_text in zip(processed_data['text'][:2], processed_data['processed_text_han'][:2]):
    print("Original:", original)
    print("HAN-ready:", han_text)
    print("-" * 50)


Original: text
HAN-ready: []
--------------------------------------------------
Original: ঢাবির ‘খ’ ইউনিট ভর্তি পরীক্ষা ঢাকা বিশ্বব্যাপী (ঢাবি) - শিক্ষাবর্ষে কলা ভুক্ত ‘খ’ ইউনিট বর্ষ সম্মান শ্রেণি ভর্তি পরীক্ষা শুক্রবার সকাল টা হওয়া যুদ্ধ চলবে বেলা টা বিশ্বব্যাপী ক্যাম্প ক্যাম্প বাই কেন্দ্র ভর্তি পরীক্ষা অনুষ্ঠিত হ বছর আসন বিপরীত শিক্ষক আবেদন করেছেন। ভর্তি পরীক্ষা উপলক্ষ্য পরীক্ষা কেন্দ্র সকাল ভিড় পরীক্ষা অভিভাব সকাল সাড়ে টা ভর্তিচ্ছু শিক্ষার্থী পরীক্ষা হওয়া প্রবেশ দেয়া দিক পরীক্ষা সুষ্ঠু সম্পন্ন পরীক্ষা হওয়া মোবাইল ফোন টেলিযোগাযোগ ইলেক্ট্রিক ডিভাইস/যন্ত্ সম্পূর্ণ নিষিদ্ধ ছাড়া পরীক্ষা চলা মোবাইল কোর্ট দায়িত্ব পালন করছেন।
HAN-ready: [['ঢাবি', 'খ', 'ইউনিট', 'ভর্তি', 'পরীক্ষা', 'ঢাকা', 'বিশ্বব্যাপী', 'ঢাবি', 'শিক্ষাবর্ষে', 'কলা', 'ভুক্ত', 'খ', 'ইউনিট', 'বর্ষ', 'সম্মান', 'শ্রেণি', 'ভর্তি', 'পরীক্ষা', 'শুক্রবা', 'সকাল', 'টা', 'হওয়া', 'যুদ্ধ', 'চলবে', 'বেলা', 'টা', 'বিশ্বব্যাপী', 'ক্যাম্প', 'ক্যাম্প', 'বা', 'কেন্দ্', 'ভর্তি', 'পরীক্ষা', 'অনুষ্ঠিত', 'হ', 'বছ', 'আসন', 'বিপরীত', 'শিক্ষক', 'আবেদ

In [2]:
processed_data.to_csv("New_han.csv", index=False)

In [3]:
import pandas as pd
new_han = pd.read_csv("New_han.csv")
def prepare_batch(text_list, word2idx, num_sents=10, num_words=20):
    encoded_docs = [ new_han(txt, word2idx, num_sents, num_words) 
                     for txt in text_list ]
    return torch.stack(encoded_docs)   # shape: (batch, num_sents, num_words)


In [2]:
import torch
import pickle
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [4]:
import os
print(os.getcwd())   # Current working directory
print(os.listdir())  # Files in this directory


c:\Users\User\Desktop\Bangla_Fake_News_detection\notebook
['bangla_bert_fake_news', 'bangla_bert_fake_news_v3', 'bangla_bert_fake_news_v4', 'bangla_data_processed.csv', 'Bert_On_newData', 'best_balanced_model.pkl', 'best_balanced_vectorizer.pkl', 'full_doc_train.csv', 'han_model.pth', 'load_data.ipynb', 'merged_dataset_predictions.csv', 'models', 'newDataset.csv', 'newfile.csv', 'New_dataset.csv', 'New_dataset_preprocessed.csv', 'New_dataset_processed.csv', 'New_han.csv', 'optimized_fake_news_model.pkl', 'predictions_on_new_data.csv', 'processed_data.csv', 'processed_data_han.csv', 'testOnNewData.ipynb', 'test_predictions_ensemble.csv', 'tfidf_vectorizer.pkl', 'tfidf_vectorizer_merged.pkl', 'tryNewData.ipynb', 'vectorizer', 'xgb_model.pkl', 'xgb_model_merged.pkl']


In [7]:
import torch
import pickle
import pandas as pd
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load vocabulary
with open("word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)


new_data = pd.read_csv("New_han.csv")  


In [8]:
new_data['processed_text_han'] = new_data['text'].apply(preprocess_for_HAN)


NameError: name 'preprocess_for_HAN' is not defined